# Classifier behavior on controlled synthetic data

This notebook is a short narrative view of the generated evidence. It reads the tabular results produced by `python -m scripts.run_experiments`; it does not recompute hidden notebook state.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
RESULTS = ROOT / 'results' / 'modern'
summary = pd.read_csv(RESULTS / 'summary.csv')
metadata = json.loads((RESULTS / 'metadata.json').read_text(encoding='utf-8'))
metadata

## Dataset geometry

Each geometry represents an explicit hypothesis. The last panel is a negative control: both labels come from the same distribution.

In [ ]:
display(Image(filename=str(RESULTS / 'dataset_geometry.png')))

## Macro F1 across models

The heatmap shows behavior under fixed conditions, not a universal model ranking. QDA can succeed on XOR and concentric structures because quadratic functions include interaction terms and covariance-driven curved boundaries.

In [ ]:
display(Image(filename=str(RESULTS / 'macro_f1_heatmap.png')))
top = (summary.sort_values(['dataset', 'macro_f1_mean'], ascending=[True, False])
              .groupby('dataset', as_index=False).head(3))
top[['dataset', 'model', 'macro_f1_mean', 'macro_f1_std', 'macro_f1_count']]

## Reading the negative control

Scores close to 0.5 are the expected outcome when features contain no information about the label. A model appearing substantially better here would be a warning about leakage, selection or sampling noise.

In [ ]:
summary.loc[summary['dataset'].eq('indistinguishable_control'),
            ['model', 'macro_f1_mean', 'macro_f1_std']].sort_values('macro_f1_mean', ascending=False)

## Limits and next experiment

The current baseline validates the dataset and comparison infrastructure. A bias-variance study must be implemented separately by fitting many models on independent training samples and evaluating them on a fixed independent population. The legacy notebook did not do this, so its original bias-variance curves are not reused.